In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_analytics"
SCHEMA = "gold"

comparison_sources = [
    "mart_executive_healthcare",
    "mart_clinical_utilization",
    "mart_claims_cost",
    "mart_hospital_operations",
    "mart_patient_outcomes",
    "mart_provider_performance",
    "mart_readmissions"
]

schema_inventory = []

for table_name in comparison_sources:
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"

    df = spark.table(full_name)

    for field in df.schema.fields:
        col_name = field.name
        data_type = field.dataType.simpleString()

        is_date_candidate = any(
            keyword in col_name.lower()
            for keyword in [
                "date",
                "time",
                "month",
                "year",
                "period",
                "start",
                "end",
                "admit",
                "discharge",
                "service"
            ]
        )

        schema_inventory.append(
            (
                table_name,
                col_name,
                data_type,
                is_date_candidate
            )
        )

schema_df = spark.createDataFrame(
    schema_inventory,
    [
        "table_name",
        "column_name",
        "data_type",
        "possible_date_field"
    ]
)

display(
    schema_df.orderBy(
        F.desc("possible_date_field"),
        "table_name",
        "column_name"
    )
)

table_name,column_name,data_type,possible_date_field
mart_claims_cost,month_start,date,true
mart_clinical_utilization,month_start,date,true
mart_executive_healthcare,eligible_inpatient_discharges,bigint,true
mart_hospital_operations,month_start,date,true
mart_patient_outcomes,birth_date,date,true
mart_patient_outcomes,death_date,date,true
mart_patient_outcomes,gender,string,true
mart_provider_performance,eligible_inpatient_discharges,bigint,true
mart_provider_performance,month_start,date,true
mart_readmissions,eligible_inpatient_discharges,bigint,true


In [0]:
from pyspark.sql import functions as F

reporting_date_sources = {
    "mart_claims_cost": "month_start",
    "mart_clinical_utilization": "month_start",
    "mart_hospital_operations": "month_start",
    "mart_provider_performance": "month_start",
    "mart_readmissions": "month_start"
}

date_profiles = []

for table_name, date_col in reporting_date_sources.items():

    full_name = f"healthcare_analytics.gold.{table_name}"
    df = spark.table(full_name)

    profile = (
        df.agg(
            F.min(F.col(date_col)).alias("min_date"),
            F.max(F.col(date_col)).alias("max_date"),
            F.countDistinct(F.col(date_col)).alias("distinct_periods"),
            F.count("*").alias("row_count")
        )
        .collect()[0]
    )

    date_profiles.append(
        (
            table_name,
            date_col,
            profile["min_date"],
            profile["max_date"],
            profile["distinct_periods"],
            profile["row_count"]
        )
    )

date_profile_df = spark.createDataFrame(
    date_profiles,
    [
        "table_name",
        "reporting_date_field",
        "min_date",
        "max_date",
        "distinct_periods",
        "row_count"
    ]
)

display(date_profile_df.orderBy("table_name"))

table_name,reporting_date_field,min_date,max_date,distinct_periods,row_count
mart_claims_cost,month_start,1921-11-01,2026-09-01,956,40354
mart_clinical_utilization,month_start,1933-04-01,2026-09-01,925,40492
mart_hospital_operations,month_start,1921-11-01,2026-09-01,956,30185
mart_provider_performance,month_start,1921-11-01,2026-09-01,956,30185
mart_readmissions,month_start,1933-11-01,2026-09-01,472,893


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# BI COMPARISON PERIOD CONTROL
# ---------------------------------------------------------

latest_observed_month = (
    date_profile_df
    .agg(F.max("max_date").alias("latest_observed_month"))
    .collect()[0]["latest_observed_month"]
)

period_control_df = (
    spark.createDataFrame(
        [(latest_observed_month,)],
        ["latest_observed_month"]
    )
    .select(
        F.col("latest_observed_month"),

        # Ignore current potentially incomplete month
        F.add_months(
            F.col("latest_observed_month"), -1
        ).alias("data_as_of_month"),

        # Current rolling 12-month period
        F.add_months(
            F.col("latest_observed_month"), -12
        ).alias("current_period_start"),

        F.add_months(
            F.col("latest_observed_month"), -1
        ).alias("current_period_end"),

        # Previous rolling 12-month period
        F.add_months(
            F.col("latest_observed_month"), -24
        ).alias("prior_period_start"),

        F.add_months(
            F.col("latest_observed_month"), -13
        ).alias("prior_period_end")
    )
)

display(period_control_df)

latest_observed_month,data_as_of_month,current_period_start,current_period_end,prior_period_start,prior_period_end
2026-09-01,2026-08-01,2025-09-01,2026-08-01,2024-09-01,2025-08-01


In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# COMPARISON PERIOD COVERAGE VALIDATION
# ---------------------------------------------------------

periods = period_control_df.collect()[0]

current_start = periods["current_period_start"]
current_end   = periods["current_period_end"]
prior_start   = periods["prior_period_start"]
prior_end     = periods["prior_period_end"]

coverage_results = []

for table_name, date_col in reporting_date_sources.items():

    df = spark.table(
        f"healthcare_analytics.gold.{table_name}"
    )

    current_df = df.filter(
        (F.col(date_col) >= F.lit(current_start)) &
        (F.col(date_col) <= F.lit(current_end))
    )

    prior_df = df.filter(
        (F.col(date_col) >= F.lit(prior_start)) &
        (F.col(date_col) <= F.lit(prior_end))
    )

    current_stats = current_df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(date_col).alias("months"),
        F.min(date_col).alias("min_month"),
        F.max(date_col).alias("max_month")
    ).collect()[0]

    prior_stats = prior_df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(date_col).alias("months"),
        F.min(date_col).alias("min_month"),
        F.max(date_col).alias("max_month")
    ).collect()[0]

    current_months = current_stats["months"]
    prior_months = prior_stats["months"]

    coverage_status = (
        "PASS"
        if current_months == 12
        and prior_months == 12
        and current_stats["rows"] > 0
        and prior_stats["rows"] > 0
        else "REVIEW"
    )

    coverage_results.append(
        (
            table_name,
            current_stats["rows"],
            current_months,
            current_stats["min_month"],
            current_stats["max_month"],
            prior_stats["rows"],
            prior_months,
            prior_stats["min_month"],
            prior_stats["max_month"],
            coverage_status
        )
    )

coverage_df = spark.createDataFrame(
    coverage_results,
    [
        "table_name",
        "current_rows",
        "current_months",
        "current_min_month",
        "current_max_month",
        "prior_rows",
        "prior_months",
        "prior_min_month",
        "prior_max_month",
        "coverage_status"
    ]
)

display(
    coverage_df.orderBy("table_name")
)

table_name,current_rows,current_months,current_min_month,current_max_month,prior_rows,prior_months,prior_min_month,prior_max_month,coverage_status
mart_claims_cost,2964,12,2025-09-01,2026-08-01,2921,12,2024-09-01,2025-08-01,PASS
mart_clinical_utilization,2202,12,2025-09-01,2026-08-01,2104,12,2024-09-01,2025-08-01,PASS
mart_hospital_operations,1962,12,2025-09-01,2026-08-01,1952,12,2024-09-01,2025-08-01,PASS
mart_provider_performance,1962,12,2025-09-01,2026-08-01,1952,12,2024-09-01,2025-08-01,PASS
mart_readmissions,43,11,2025-09-01,2026-08-01,36,12,2024-09-01,2025-08-01,REVIEW


In [0]:
display(
    coverage_df.select(
        "table_name",
        "current_months",
        "prior_months",
        "current_rows",
        "prior_rows",
        "coverage_status"
    ).orderBy("table_name")
)

table_name,current_months,prior_months,current_rows,prior_rows,coverage_status
mart_claims_cost,12,12,2964,2921,PASS
mart_clinical_utilization,12,12,2202,2104,PASS
mart_hospital_operations,12,12,1962,1952,PASS
mart_provider_performance,12,12,1962,1952,PASS
mart_readmissions,11,12,43,36,REVIEW


In [0]:
from pyspark.sql import functions as F

readm = spark.table(
    "healthcare_analytics.gold.mart_readmissions"
)

current_readm_months = (
    readm
    .filter(
        (F.col("month_start") >= F.lit(current_start)) &
        (F.col("month_start") <= F.lit(current_end))
    )
    .groupBy("month_start")
    .agg(
        F.count("*").alias("row_count"),
        F.sum("readmissions_30d").alias("readmissions_30d"),
        F.sum("eligible_inpatient_discharges").alias(
            "eligible_inpatient_discharges"
        )
    )
    .orderBy("month_start")
)

display(current_readm_months)

month_start,row_count,readmissions_30d,eligible_inpatient_discharges
2025-09-01,3,2,4
2025-10-01,2,1,5
2025-11-01,4,1,4
2025-12-01,6,1,7
2026-01-01,3,1,3
2026-03-01,4,1,4
2026-04-01,4,0,5
2026-05-01,6,0,6
2026-06-01,4,1,5
2026-07-01,3,0,3


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, TimestampType

# ---------------------------------------------------------
# DIAGNOSE MISSING FEBRUARY 2026 READMISSION MONTH
# ---------------------------------------------------------

diagnostic_results = []

for table_name in ["fact_readmission", "fact_encounter"]:

    full_name = f"healthcare_analytics.gold.{table_name}"
    df = spark.table(full_name)

    date_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, (DateType, TimestampType))
    ]

    for date_col in date_columns:

        feb_rows = (
            df
            .filter(
                F.to_date(F.col(date_col)).between(
                    "2026-02-01",
                    "2026-02-28"
                )
            )
            .count()
        )

        diagnostic_results.append(
            (
                table_name,
                date_col,
                feb_rows
            )
        )

feb_diagnostic_df = spark.createDataFrame(
    diagnostic_results,
    [
        "table_name",
        "date_column",
        "feb_2026_rows"
    ]
)

display(
    feb_diagnostic_df.orderBy(
        F.desc("feb_2026_rows"),
        "table_name",
        "date_column"
    )
)

table_name,date_column,feb_2026_rows
fact_encounter,encounter_start,345
fact_encounter,encounter_end,342
fact_readmission,encounter_start,2
fact_readmission,next_inpatient_start,2
fact_encounter,_silver_processed_at,0
fact_encounter,ingested_at,0
fact_readmission,encounter_end,0


In [0]:
from pyspark.sql import functions as F

fr = spark.table("healthcare_analytics.gold.fact_readmission")

# Pull useful readmission-related columns dynamically
interesting_cols = [
    c for c in fr.columns
    if any(
        keyword in c.lower()
        for keyword in [
            "patient",
            "encounter",
            "start",
            "end",
            "next",
            "readmission",
            "eligible",
            "organization",
            "provider"
        ]
    )
]

feb_readmission_rows = (
    fr
    .filter(
        (F.to_date(F.col("encounter_start")) >= F.lit("2026-02-01")) &
        (F.to_date(F.col("encounter_start")) <= F.lit("2026-02-28"))
    )
    .select(*interesting_cols)
    .orderBy("encounter_start")
)

display(feb_readmission_rows)

encounter_id,patient_id,organization_id,provider_id,encounter_start,encounter_end,patient_responsibility,next_inpatient_encounter_id,next_inpatient_start,days_to_readmission,readmission_30d_flag,index_encounter_key,patient_key,provider_key
08bd3520-7430-dcb8-f504-c20ebc123801,08bd3520-7430-dcb8-ce26-21a9cfafa46f,38508743-50d1-3429-8fe2-37e42ddaf20e,d5a3e338-3a78-3bb9-af27-324e7421202a,2026-02-20T15:53:02.000Z,2026-03-03T16:34:02.000Z,22098.020000000004,08bd3520-7430-dcb8-354d-832500109d87,2026-04-02T04:34:02.000Z,30,1,05e2d02f7d7eec552eb327468d05e96878ca95fb25797988ea64b1c21148e877,51811cd7682ba7a098bbe34f761b762f20af75ca15151f949c4d8e12c64e2e18,c6318dd4a616f8bf04bcf94b8f22503a5229de71924b1be93fc0224089b9c648
370c536b-02f0-a1e7-e3bc-4c82d6bea715,370c536b-02f0-a1e7-adf7-7ada47cc7b9c,497f39dd-280e-3d58-af5b-c5e3a3a09b10,461bab1e-7c0e-3c41-bd09-f2d74fa8bfee,2026-02-28T10:41:56.000Z,2026-03-16T10:41:56.000Z,29.24000000000001,null,null,null,0,58937802dc1e263fd33eacb71058dd14fece59fbe291f643e17177634054fa41,b5909d8da744fcfb1b3199a188757c8efcbc8ec93a7cf4fd8c6ab0ad306bd811,e1ee34c749e31a766cd20479b73c3344776cbdd493223d8efafb28b36d1ee305


In [0]:
from pyspark.sql import functions as F

fr = spark.table("healthcare_analytics.gold.fact_readmission")

candidate_cols = [
    "encounter_id",
    "patient_id",
    "organization_id",
    "provider_id",

    "encounter_start",
    "encounter_end",
    "next_inpatient_start",

    "days_to_readmission",
    "readmission_days",

    "readmission_30d",
    "is_readmission_30d",

    "eligible_inpatient_discharge",
    "eligible_inpatient_discharges",
    "is_eligible_inpatient_discharge",

    "encounter_class",
    "encounter_type"
]

existing_cols = [
    c for c in candidate_cols
    if c in fr.columns
]

print("Columns selected:")
print(existing_cols)

feb_detail = (
    fr
    .filter(
        F.to_date("encounter_start").between(
            "2026-02-01",
            "2026-02-28"
        )
    )
    .select(*existing_cols)
    .orderBy("encounter_start")
)

display(feb_detail)

Columns selected:
['encounter_id', 'patient_id', 'organization_id', 'provider_id', 'encounter_start', 'encounter_end', 'next_inpatient_start', 'days_to_readmission']


encounter_id,patient_id,organization_id,provider_id,encounter_start,encounter_end,next_inpatient_start,days_to_readmission
08bd3520-7430-dcb8-f504-c20ebc123801,08bd3520-7430-dcb8-ce26-21a9cfafa46f,38508743-50d1-3429-8fe2-37e42ddaf20e,d5a3e338-3a78-3bb9-af27-324e7421202a,2026-02-20T15:53:02.000Z,2026-03-03T16:34:02.000Z,2026-04-02T04:34:02.000Z,30
370c536b-02f0-a1e7-e3bc-4c82d6bea715,370c536b-02f0-a1e7-adf7-7ada47cc7b9c,497f39dd-280e-3d58-af5b-c5e3a3a09b10,461bab1e-7c0e-3c41-bd09-f2d74fa8bfee,2026-02-28T10:41:56.000Z,2026-03-16T10:41:56.000Z,null,null


In [0]:
from pyspark.sql import functions as F

# =========================================================
# READMISSION BI MONTHLY SERIES
# Guarantees every reporting month is represented
# =========================================================

fr = spark.table(
    "healthcare_analytics.gold.fact_readmission"
)

# Full month spine for the two comparison windows
month_spine = spark.sql("""
SELECT explode(
    sequence(
        to_date('2024-09-01'),
        to_date('2026-08-01'),
        interval 1 month
    )
) AS month_start
""")

# Aggregate actual readmission fact data
readmission_monthly_actual = (
    fr
    .withColumn(
        "month_start",
        F.trunc(F.to_date("encounter_start"), "month")
    )
    .filter(
        F.col("month_start").between(
            "2024-09-01",
            "2026-08-01"
        )
    )
    .groupBy("month_start")
    .agg(
        F.count("*").alias(
            "eligible_inpatient_discharges"
        ),

        F.sum(
            F.when(
                (F.col("days_to_readmission") >= 0) &
                (F.col("days_to_readmission") <= 30),
                1
            ).otherwise(0)
        ).alias("readmissions_30d")
    )
)

# Join to month spine so missing months cannot disappear
bi_readmissions_monthly = (
    month_spine
    .join(
        readmission_monthly_actual,
        "month_start",
        "left"
    )
    .fillna(
        {
            "eligible_inpatient_discharges": 0,
            "readmissions_30d": 0
        }
    )
    .withColumn(
        "readmission_rate_pct",
        F.when(
            F.col("eligible_inpatient_discharges") > 0,
            F.round(
                F.col("readmissions_30d") /
                F.col("eligible_inpatient_discharges") * 100,
                2
            )
        )
    )
    .orderBy("month_start")
)

display(
    bi_readmissions_monthly.filter(
        F.col("month_start").between(
            "2026-01-01",
            "2026-03-01"
        )
    )
)

month_start,eligible_inpatient_discharges,readmissions_30d,readmission_rate_pct
2026-01-01,3,1,33.33
2026-02-01,2,1,50.0
2026-03-01,2,0,0.0


In [0]:
from pyspark.sql import functions as F

# ============================================================
# EXECUTIVE KPI COMPARISON SERVING LAYER
# Current 12M vs Prior 12M
# ============================================================

# ------------------------------------------------------------
# PERIODS
# ------------------------------------------------------------

p = period_control_df.collect()[0]

current_start = p["current_period_start"]
current_end   = p["current_period_end"]
prior_start   = p["prior_period_start"]
prior_end     = p["prior_period_end"]


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def pct_change(current, prior):
    if prior in (None, 0):
        return None
    return round(((current - prior) / prior) * 100, 2)


def direction(current, prior):
    if current is None or prior is None:
        return "NA"
    if current > prior:
        return "UP"
    if current < prior:
        return "DOWN"
    return "FLAT"


def arrow(current, prior):
    d = direction(current, prior)

    if d == "UP":
        return "▲"
    elif d == "DOWN":
        return "▼"
    elif d == "FLAT":
        return "●"
    return ""


def count_display(v):
    return f"{int(v):,}"


def pct_display(v):
    return f"{v:.2f}%"


def money_display(v):
    if v >= 1_000_000:
        return f"${v / 1_000_000:.1f}M"
    elif v >= 1_000:
        return f"${v / 1_000:.1f}K"
    return f"${v:,.0f}"


def resolve_column(df, candidates, label):
    lower_map = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    raise ValueError(
        f"Could not find {label}. "
        f"Available columns: {df.columns}"
    )


# ============================================================
# 1. PATIENT / ENCOUNTER KPIs
# ============================================================

enc = spark.table(
    "healthcare_analytics.gold.fact_encounter"
)

encounter_class_col = resolve_column(
    enc,
    [
        "encounter_class",
        "encounterclass",
        "class",
        "encounter_type"
    ],
    "encounter class field"
)


def encounter_metrics(start_date, end_date):

    df = enc.filter(
        F.to_date("encounter_start").between(
            F.lit(start_date),
            F.lit(end_date)
        )
    )

    result = (
        df.agg(
            F.countDistinct("patient_id")
             .alias("patients"),

            F.countDistinct("encounter_id")
             .alias("encounters"),

            F.countDistinct(
                F.when(
                    F.lower(F.col(encounter_class_col))
                     .contains("inpatient"),
                    F.col("encounter_id")
                )
            ).alias("inpatient_encounters")
        )
        .collect()[0]
    )

    return {
        "patients": float(result["patients"] or 0),
        "encounters": float(result["encounters"] or 0),
        "inpatient_encounters":
            float(result["inpatient_encounters"] or 0)
    }


current_enc = encounter_metrics(
    current_start,
    current_end
)

prior_enc = encounter_metrics(
    prior_start,
    prior_end
)


# ============================================================
# 2. CLAIM / COST KPIs
# ============================================================

claims = spark.table(
    "healthcare_analytics.gold.mart_claims_cost"
)

claim_cost_col = resolve_column(
    claims,
    [
        "total_claim_cost",
        "total_claim_amount",
        "claim_cost",
        "total_cost"
    ],
    "total claim cost field"
)

claim_encounter_col = resolve_column(
    claims,
    [
        "claim_encounter_count",
        "total_encounters",
        "encounter_count",
        "encounters",
        "distinct_encounters"
    ],
    "claim encounter count field"
)


def claim_metrics(start_date, end_date):

    row = (
        claims
        .filter(
            F.col("month_start").between(
                F.lit(start_date),
                F.lit(end_date)
            )
        )
        .agg(
            F.sum(F.col(claim_cost_col))
             .alias("claim_cost"),

            F.sum(F.col(claim_encounter_col))
             .alias("encounters")
        )
        .collect()[0]
    )

    total_cost = float(row["claim_cost"] or 0)
    encounter_count = float(row["encounters"] or 0)

    avg_cost = (
        total_cost / encounter_count
        if encounter_count > 0
        else 0
    )

    return total_cost, avg_cost


current_claim_cost, current_avg_cost = claim_metrics(
    current_start,
    current_end
)

prior_claim_cost, prior_avg_cost = claim_metrics(
    prior_start,
    prior_end
)


# ============================================================
# 3. READMISSION KPI
# ============================================================

def readmission_metrics(start_date, end_date):

    row = (
        bi_readmissions_monthly
        .filter(
            F.col("month_start").between(
                F.lit(start_date),
                F.lit(end_date)
            )
        )
        .agg(
            F.sum("readmissions_30d")
             .alias("readmissions"),

            F.sum("eligible_inpatient_discharges")
             .alias("eligible")
        )
        .collect()[0]
    )

    readmissions = float(row["readmissions"] or 0)
    eligible = float(row["eligible"] or 0)

    rate = (
        readmissions / eligible * 100
        if eligible > 0
        else 0
    )

    return rate


current_readmission_rate = readmission_metrics(
    current_start,
    current_end
)

prior_readmission_rate = readmission_metrics(
    prior_start,
    prior_end
)


# ============================================================
# KPI RECORD BUILDER
# ============================================================

records = []


def add_kpi(
    sort_order,
    kpi_name,
    current_value,
    prior_value,
    display_type,
    favorable_rule="NEUTRAL"
):

    delta = current_value - prior_value
    change_pct = pct_change(
        current_value,
        prior_value
    )

    d = direction(
        current_value,
        prior_value
    )

    # ----------------------------------
    # Performance signal
    # ----------------------------------

    if favorable_rule == "LOWER":

        if d == "DOWN":
            signal = "FAVORABLE"
        elif d == "UP":
            signal = "UNFAVORABLE"
        else:
            signal = "NEUTRAL"

    elif favorable_rule == "HIGHER":

        if d == "UP":
            signal = "FAVORABLE"
        elif d == "DOWN":
            signal = "UNFAVORABLE"
        else:
            signal = "NEUTRAL"

    else:
        signal = "NEUTRAL"

    # ----------------------------------
    # Display formatting
    # ----------------------------------

    if display_type == "COUNT":
        current_display = count_display(
            current_value
        )
        prior_display = count_display(
            prior_value
        )

    elif display_type == "PERCENT":
        current_display = pct_display(
            current_value
        )
        prior_display = pct_display(
            prior_value
        )

    elif display_type == "MONEY":
        current_display = money_display(
            current_value
        )
        prior_display = money_display(
            prior_value
        )

    # ----------------------------------
    # Comparison text
    # ----------------------------------

    if display_type == "PERCENT":

        comparison_text = (
            f"{arrow(current_value, prior_value)} "
            f"{abs(delta):.2f} pp vs prior 12M"
        )

    else:

        comparison_text = (
            f"{arrow(current_value, prior_value)} "
            f"{abs(change_pct):.1f}% vs prior 12M"
            if change_pct is not None
            else "No prior-period comparison"
        )

    records.append(
        (
            "Executive Overview",
            sort_order,
            kpi_name,
            float(current_value),
            float(prior_value),
            float(delta),
            float(change_pct)
                if change_pct is not None
                else None,
            d,
            signal,
            current_display,
            prior_display,
            comparison_text,
            current_start,
            current_end,
            prior_start,
            prior_end
        )
    )


# ============================================================
# SIX EXECUTIVE KPIs
# ============================================================

add_kpi(
    1,
    "Total Patients",
    current_enc["patients"],
    prior_enc["patients"],
    "COUNT",
    "NEUTRAL"
)

add_kpi(
    2,
    "Total Encounters",
    current_enc["encounters"],
    prior_enc["encounters"],
    "COUNT",
    "NEUTRAL"
)

add_kpi(
    3,
    "Total Inpatient Encounters",
    current_enc["inpatient_encounters"],
    prior_enc["inpatient_encounters"],
    "COUNT",
    "NEUTRAL"
)

add_kpi(
    4,
    "Readmission Rate %",
    current_readmission_rate,
    prior_readmission_rate,
    "PERCENT",
    "LOWER"
)

add_kpi(
    5,
    "Avg Cost Per Encounter",
    current_avg_cost,
    prior_avg_cost,
    "MONEY",
    "LOWER"
)

add_kpi(
    6,
    "Total Claim Cost",
    current_claim_cost,
    prior_claim_cost,
    "MONEY",
    "NEUTRAL"
)


# ============================================================
# CREATE SERVING TABLE
# ============================================================

kpi_comparison_df = spark.createDataFrame(
    records,
    [
        "dashboard_page",
        "sort_order",
        "kpi_name",
        "current_value",
        "prior_value",
        "delta_value",
        "delta_pct",
        "direction",
        "performance_signal",
        "current_display",
        "prior_display",
        "comparison_text",
        "current_period_start",
        "current_period_end",
        "prior_period_start",
        "prior_period_end"
    ]
)

(
    kpi_comparison_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "healthcare_analytics.gold.bi_kpi_comparison"
    )
)

display(
    kpi_comparison_df.orderBy("sort_order")
)

dashboard_page,sort_order,kpi_name,current_value,prior_value,delta_value,delta_pct,direction,performance_signal,current_display,prior_display,comparison_text,current_period_start,current_period_end,prior_period_start,prior_period_end
Executive Overview,1,Total Patients,914.0,906.0,8.0,0.88,UP,NEUTRAL,914,906,▲ 0.9% vs prior 12M,2025-09-01,2026-08-01,2024-09-01,2025-08-01
Executive Overview,2,Total Encounters,4435.0,4452.0,-17.0,-0.38,DOWN,NEUTRAL,"4,435","4,452",▼ 0.4% vs prior 12M,2025-09-01,2026-08-01,2024-09-01,2025-08-01
Executive Overview,3,Total Inpatient Encounters,46.0,42.0,4.0,9.52,UP,NEUTRAL,46,42,▲ 9.5% vs prior 12M,2025-09-01,2026-08-01,2024-09-01,2025-08-01
Executive Overview,4,Readmission Rate %,15.686274509803921,18.181818181818183,-2.4955436720142625,-13.73,DOWN,FAVORABLE,15.69%,18.18%,▼ 2.50 pp vs prior 12M,2025-09-01,2026-08-01,2024-09-01,2025-08-01
Executive Overview,5,Avg Cost Per Encounter,3043.9274001663894,2845.8951887382723,198.03221142811708,6.96,UP,UNFAVORABLE,$3.0K,$2.8K,▲ 7.0% vs prior 12M,2025-09-01,2026-08-01,2024-09-01,2025-08-01
Executive Overview,6,Total Claim Cost,1.463520294E7,1.3646067430000015E7,989135.5099999849,7.25,UP,NEUTRAL,$14.6M,$13.6M,▲ 7.2% vs prior 12M,2025-09-01,2026-08-01,2024-09-01,2025-08-01


In [0]:
tables_to_check = [
    "mart_claims_cost",
    "mart_clinical_utilization",
    "mart_hospital_operations",
    "mart_patient_outcomes",
    "mart_provider_performance",
    "mart_readmissions"
]

for table_name in tables_to_check:
    print("\n" + "=" * 90)
    print(table_name)
    print("=" * 90)

    df = spark.table(f"healthcare_analytics.gold.{table_name}")

    print(df.columns)


mart_claims_cost
['month_start', 'payer_key', 'payer_name', 'facility_key', 'organization_name', 'state', 'claim_encounter_count', 'unique_patients', 'total_claim_cost', 'avg_claim_cost', 'payer_coverage', 'patient_responsibility', 'payer_coverage_pct']

mart_clinical_utilization
['month_start', 'code', 'description', 'event_count', 'unique_patients', 'unique_encounters', 'clinical_event_type', 'total_event_cost']

mart_hospital_operations
['facility_key', 'organization_name', 'city', 'state', 'month_start', 'encounter_count', 'unique_patients', 'inpatient_encounters', 'emergency_encounters', 'ambulatory_encounters', 'avg_encounter_duration_hours', 'total_claim_cost', 'avg_cost_per_encounter', 'payer_coverage', 'patient_responsibility', 'inpatient_pct']

mart_patient_outcomes
['patient_id', 'patient_name', 'birth_date', 'death_date', 'age_at_reference', 'age_band', 'gender', 'race', 'ethnicity', 'marital', 'city', 'state', 'county', 'zip', 'lat', 'lon', 'healthcare_expenses', 'healthc

In [0]:
common_filter_candidates = [
    "month_start",
    "facility_key",
    "organization_name",
    "city",
    "state",
    "payer_key",
    "payer_name",
    "provider_key",
    "provider_name",
    "gender"
]

tables_to_check = [
    "mart_claims_cost",
    "mart_clinical_utilization",
    "mart_hospital_operations",
    "mart_patient_outcomes",
    "mart_provider_performance",
    "mart_readmissions"
]

rows = []

for table_name in tables_to_check:
    cols = set(
        spark.table(
            f"healthcare_analytics.gold.{table_name}"
        ).columns
    )

    row = {"table_name": table_name}

    for field in common_filter_candidates:
        row[field] = "YES" if field in cols else "—"

    rows.append(row)

filter_matrix_df = spark.createDataFrame(rows)

display(filter_matrix_df)

city,facility_key,gender,month_start,organization_name,payer_key,payer_name,provider_key,provider_name,state,table_name
—,YES,—,YES,YES,YES,YES,—,—,YES,mart_claims_cost
—,—,—,YES,—,—,—,—,—,—,mart_clinical_utilization
YES,YES,—,YES,YES,—,—,—,—,YES,mart_hospital_operations
YES,—,YES,—,—,—,—,—,—,YES,mart_patient_outcomes
—,YES,—,YES,YES,—,—,YES,YES,—,mart_provider_performance
YES,YES,—,YES,YES,—,—,—,—,YES,mart_readmissions


In [0]:
from pyspark.sql import functions as F

(
    spark.range(1)
    .select(
        F.current_timestamp().alias("dashboard_refreshed_at")
    )
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "healthcare_analytics.gold.bi_dashboard_refresh"
    )
)

display(
    spark.table(
        "healthcare_analytics.gold.bi_dashboard_refresh"
    )
)

dashboard_refreshed_at
2026-09-10T08:12:17.465Z
